In [2]:
# pip install qcd-gym gymnasium stable-baselines3 torch numpy
# 可选：pip install matplotlib

import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy

In [3]:
# 1) 创建环境：1 比特、最大深度 6，目标是 “Unitary Composition: Hadamard”
#    render_mode="text" 便于之后查看学到的门序列
env = gym.make(
    "CircuitDesigner-v0",
    max_qubits=1,
    max_depth=6,
    objective="UC-hadamard",
    render_mode="text",
)

# SB3 需要 Monitor 包装以记录回合统计
env = Monitor(env)

# 2) 用 PPO 训练（UC-hadamard 很简单，几万步就能收敛）
model = PPO(
    "MlpPolicy",
    env,
    n_steps=2048,
    batch_size=256,
    gae_lambda=0.95,
    gamma=0.995,
    learning_rate=3e-4,
    ent_coef=0.0,
    verbose=1,
    device="auto",
)
model.learn(total_timesteps=50_000)

# 3) 评估：回合奖励越接近 1 越好（回合末: 1 - arctan(||U - V||_F) 再减深度惩罚）:contentReference[oaicite:1]{index=1}
mean_r, std_r = evaluate_policy(model, env, n_eval_episodes=20, deterministic=True)
print(f"Mean reward over 20 episodes: {mean_r:.4f} ± {std_r:.4f}")

# 4) 展示一次“学到的”门序列（非硬编码）
#    按 qcd-gym 的动作编码： o=0 为 PhaseShift/ControlledPhaseShift, o=1 为 RX/CNOT, o=2 为 Terminate :contentReference[oaicite:2]{index=2}
def decode_action(a):
    o, q, c, phi = a
    if o == 0:
        return f"P({phi:+.3f}) on q={int(q)}"
    if o == 1 and int(q) == int(c):
        return f"RX({phi:+.3f}) on q={int(q)}"
    if o == 1 and int(q) != int(c):
        return f"CNOT q={int(q)} -> c={int(c)}"
    if o == 2:
        return "Terminate"
    return f"Unknown({a})"

obs, info = env.reset(seed=0)
done = False
steps = 0
actions_log = []

while not done:
    action, _ = model.predict(obs, deterministic=True)
    actions_log.append(action.copy())
    obs, reward, terminated, truncated, info = env.step(action)
    steps += 1
    if terminated or truncated:
        done = True

# 打印动作序列（学到的 UC-H 门分解），并渲染电路文本
print("\nLearned action sequence:")
for i, a in enumerate(actions_log, 1):
    print(f" {i:02d}. {decode_action(a)}")
print("\nEpisode return:", reward)

# 文本渲染（需创建环境时 render_mode='text'）
try:
    env.render()
except Exception as e:
    print("(render not available:", e, ")")

env.close()





Using cpu device
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 5.64     |
|    ep_rew_mean     | -0.0398  |
| time/              |          |
|    fps             | 1330     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 5.72        |
|    ep_rew_mean          | -0.0455     |
| time/                   |             |
|    fps                  | 1261        |
|    iterations           | 2           |
|    time_elapsed         | 3           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.006596473 |
|    clip_fraction        | 0.0741      |
|    clip_range           | 0.2         |
|    entropy_loss         | -5.67       |
|    explained_varia